# 12. Cleaning: Online Retail II (UCI)

The raw file `Online Retail II UCI/online_retail_II.csv` is read, and the cleaned tables are written to `clean/online_retail_ii__*.csv` with the same files and columns as the dunnhumby notebook.

The dataset has no category column, so items are SKUs (`StockCode`).

Design choices:
- item = `StockCode` (4,618 products)
- customer = `Customer ID`
- period = calendar month, keep Dec 2009 - Nov 2011 (periods 0-23)
- edges are binary
- split: train 0-15, validation 16-19, test 20-23
- features: Country one-hot only (about 90% of customers are in the UK), so there are practically no customer features

In [1]:
import numpy as np, pandas as pd, json
from pathlib import Path

SLUG  = "online_retail_ii"
SRC   = next(p for p in [Path("online_retail_II.csv"),
                         Path("Online Retail II UCI/online_retail_II.csv")] if p.exists())
OUT   = Path("clean"); OUT.mkdir(exist_ok=True)

KEEP_PERIODS = (0, 23)      # drop Dec-2011: only 8 trading days, halves the test window
CYCLE_LEN    = 12
SPLIT        = {"train": range(0, 16), "val": (16, 17, 18, 19), "test": (20, 21, 22, 23)}
SEED = 42; np.random.seed(SEED)
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)

LEDGER = []
def note(step, n, detail=""):
    prev = LEDGER[-1][1] if LEDGER else n
    LEDGER.append((step, n, n - prev, detail))
    print(f"{step:<46} rows={n:>9,} ({n-prev:+,})  {detail}")

raw = pd.read_csv(SRC)
raw.columns = [c.strip().replace(" ", "_") for c in raw.columns]
raw["dt"] = pd.to_datetime(raw.InvoiceDate)
note("00  raw file", len(raw), f"{raw.Customer_ID.nunique():,} customers, {raw.dt.min().date()} to {raw.dt.max().date()}")

00  raw file                                   rows=1,067,371 (+0)  5,942 customers, 2009-12-01 to 2011-12-09


## 1. Cleaning steps
Six filters; each prints what it removes.

In [2]:
d = raw.copy()
sc = d.StockCode.astype(str)

# D1: credit invoices are returns
d = d[~d.Invoice.astype(str).str.startswith("C")]
note("01  dropped cancellations", len(d))

# D2: a row with no customer cannot be a graph node
d = d[d.Customer_ID.notna()]
note("02  dropped rows with no Customer ID", len(d), f"{(1-len(d)/len(raw))*100:.1f}% of file gone so far")

# D3: postage, fees, adjustments, bank charges: not products.
# Anchor at the START only: "85123A" and "84406B" are real colour/size variants of a 5-digit
# product code and must be kept; "POST", "DOT", "BANK CHARGES", "ADJUST", "M", "D", "C2",
# "AMAZONFEE", "CRUK" and "DCGS*" do not start with five digits and are dropped.
sc = d.StockCode.astype(str)
d = d[sc.str.match(r"^\d{5}")]
note("03  dropped non-product stock codes", len(d), f"{d.StockCode.nunique():,} products remain")

# D4: invalid values
d = d[(d.Quantity > 0) & (d.Price > 0)]
note("04  dropped invalid quantity / price", len(d))

# D5: calendar month index; drop the 8-day December 2011 stub
per = d.dt.dt.to_period("M")
d["period_idx"] = (per - raw.dt.dt.to_period("M").min()).apply(lambda x: x.n)
lo, hi = KEEP_PERIODS
d = d[d.period_idx.between(lo, hi)].copy()
note("05  dropped partial final month", len(d), f"periods {d.period_idx.min()}–{d.period_idx.max()}")

# D6: duplicate line rows. About a quarter of the multi-row (invoice, item) groups carry
# genuinely different quantities, so summing is correct where dropping would lose real units.
before = len(d)
d["Customer_ID"] = d.Customer_ID.astype("int64")
d = (d.groupby(["Invoice","StockCode","Customer_ID","period_idx","dt","Country"], as_index=False)
       .agg(quantity=("Quantity","sum"), price=("Price","mean")))
note("06  collapsed duplicate line rows", len(d), f"{before-len(d):,} rows merged, quantity summed")

d["revenue"] = (d.quantity * d.price).round(2)
d["Country"] = d.Country.astype("string").str.strip()
g = d.groupby("Invoice")
assert g.Customer_ID.nunique().max() == 1 and g.dt.apply(lambda s: s.dt.date.nunique()).max() == 1
print(f"    {d.Invoice.nunique():,} invoices, each one customer and one day")

01  dropped cancellations                      rows=1,047,877 (-19,494)  
02  dropped rows with no Customer ID           rows=  805,620 (-242,257)  24.5% of file gone so far


03  dropped non-product stock codes            rows=  802,693 (-2,927)  4,619 products remain
04  dropped invalid quantity / price           rows=  802,632 (-61)  


05  dropped partial final month                rows=  785,390 (-17,242)  periods 0–23


06  collapsed duplicate line rows              rows=  749,476 (-35,914)  35,914 rows merged, quantity summed


    35,818 invoices, each one customer and one day


## 2. Calendar
Real dates are available and cover two Christmas peaks.

In [3]:
split_of = {p: name for name, rng in SPLIT.items() for p in rng}
periods  = sorted(d.period_idx.unique())
labels   = (d.groupby("period_idx").dt.min().dt.to_period("M").astype(str)).reindex(periods)
cycle    = labels.str.slice(5, 7).astype(int).values            # calendar month 1–12

calendar = pd.DataFrame({
    "period_idx": periods, "label": labels.values, "cycle_pos": cycle,
    "cycle_sin": np.sin(2*np.pi*cycle/CYCLE_LEN).round(6),
    "cycle_cos": np.cos(2*np.pi*cycle/CYCLE_LEN).round(6),
    "elapsed":   periods,
    "split":     [split_of[p] for p in periods]})
print(calendar.to_string(index=False))

 period_idx   label  cycle_pos  cycle_sin  cycle_cos  elapsed split
          0 2009-12         12  -0.000000   1.000000        0 train
          1 2010-01          1   0.500000   0.866025        1 train
          2 2010-02          2   0.866025   0.500000        2 train
          3 2010-03          3   1.000000   0.000000        3 train
          4 2010-04          4   0.866025  -0.500000        4 train
          5 2010-05          5   0.500000  -0.866025        5 train
          6 2010-06          6   0.000000  -1.000000        6 train
          7 2010-07          7  -0.500000  -0.866025        7 train
          8 2010-08          8  -0.866025  -0.500000        8 train
          9 2010-09          9  -1.000000  -0.000000        9 train
         10 2010-10         10  -0.866025   0.500000       10 train
         11 2010-11         11  -0.500000   0.866025       11 train
         12 2010-12         12  -0.000000   1.000000       12 train
         13 2011-01          1   0.500000   0.86

## 3. Integer indices (same as dunnhumby)

In [4]:
def remap(values, idx_col, label_col):
    """Sorted unique values -> 0..n-1. Sorting makes the mapping deterministic across runs."""
    uniq = sorted(pd.Series(values).dropna().unique())
    return {v: i for i, v in enumerate(uniq)}, pd.DataFrame({idx_col: range(len(uniq)), label_col: uniq})

cust_map, customer_id_map = remap(d.Customer_ID, "customer_idx", "customer_id")
item_map,  item_id_map    = remap(d.StockCode,   "item_idx",     "item_label")
d["customer_idx"] = d.Customer_ID.map(cust_map).astype("int32")
d["item_idx"]     = d.StockCode.map(item_map).astype("int32")
N_CUST, N_ITEM = len(cust_map), len(item_map)
d["item_node_id"] = d.item_idx + N_CUST

# modal description per stock code: 1,232 codes carry more than one spelling
desc = (raw.dropna(subset=["Description"]).groupby("StockCode").Description
          .agg(lambda s: s.mode().iat[0]).rename("description"))
item_id_map = item_id_map.join(desc, on="item_label")
item_id_map["parent_department"] = pd.NA      # no taxonomy in this dataset — kept for schema parity
item_id_map["parent_commodity"]  = pd.NA
item_id_map = item_id_map[["item_idx","item_label","description","parent_department","parent_commodity"]]

print(f"customers {N_CUST:,} -> idx 0..{N_CUST-1}")
print(f"items     {N_ITEM:,} -> idx 0..{N_ITEM-1}  (node ids {N_CUST}..{N_CUST+N_ITEM-1})")
assert (d.customer_idx.map(dict(zip(customer_id_map.customer_idx, customer_id_map.customer_id)))
        == d.Customer_ID).all()
item_id_map.head(3)

customers 5,824 -> idx 0..5823
items     4,618 -> idx 0..4617  (node ids 5824..10441)


,item_idx,item_label,description,parent_department,parent_commodity
0,0,10002,INFLATABLE POLITICAL GLOBE,<NA>,<NA>
1,1,10080,GROOVY CACTUS INFLATABLE,<NA>,<NA>
2,2,10109,BENDY COLOUR PENCILS,<NA>,<NA>


## 4. Edge list

In [5]:
interactions = (d.groupby(["period_idx","customer_idx","item_idx"], observed=True)
                  .agg(n_lineitems=("quantity","size"), n_baskets=("Invoice","nunique"),
                       total_quantity=("quantity","sum"), total_revenue=("revenue","sum"))
                  .reset_index())
interactions["total_revenue"] = interactions.total_revenue.round(2)
interactions["log_quantity"]  = np.log1p(interactions.total_quantity).round(4)
interactions["weight"]        = 1

print(f"{len(d):,} line items -> {len(interactions):,} edges")
print(f"{len(interactions)//len(periods):,} edges per graph over {len(periods)} periods")
interactions.head(3)

749,476 line items -> 702,450 edges
29,268 edges per graph over 24 periods


,period_idx,customer_idx,item_idx,n_lineitems,n_baskets,total_quantity,total_revenue,log_quantity,weight
0,0,12,22,1,1,60,297.0,4.1109,1
1,0,12,23,1,1,60,297.0,4.1109,1
2,0,12,24,1,1,30,178.5,3.4340,1


## 5. Customer features
Only Country is available, and about 90% of customers are in the UK. The features are exported so the schema matches the dunnhumby files; `has_demographics` is 0 for everyone.

In [6]:
home = (d.sort_values("dt").drop_duplicates("customer_idx")
          .set_index("customer_idx").Country.rename("country"))
cf = customer_id_map.join(home, on="customer_idx")
cf["has_demographics"] = 0

TOP = cf.country.value_counts().head(8).index           # rarer countries fold into "Other"
cf["country_grp"] = np.where(cf.country.isin(TOP), cf.country, "Other")
onehot = pd.get_dummies(cf.country_grp.astype(str), prefix="country", dtype="int8")
FEATURE_COLS = list(onehot.columns)
cf = pd.concat([cf, onehot], axis=1)

activity = (interactions.groupby("customer_idx")
              .agg(first_period_idx=("period_idx","min"), last_period_idx=("period_idx","max"),
                   n_active_periods=("period_idx","nunique")).reset_index())
customer_features = cf.merge(activity, on="customer_idx").sort_values("customer_idx")

print(f"{len(customer_features):,} customers | feature matrix: {len(FEATURE_COLS)} dimensions")
print(f"UK share: {(cf.country=='United Kingdom').mean()*100:.1f}%  "
      f"-> the feature matrix is near-constant; treat as 'no features'")
print(cf.country_grp.value_counts().to_string())

5,824 customers | feature matrix: 9 dimensions
UK share: 91.2%  -> the feature matrix is near-constant; treat as 'no features'
country_grp
United Kingdom    5313
Other              187
Germany            106
France              90
Spain               35
Belgium             27
Portugal            23
Netherlands         22
Switzerland         21


## 6. Leakage checks (same four as dunnhumby)

In [7]:
# G1: every edge's activity lies inside its own period
chk = d.groupby(["period_idx","customer_idx","item_idx"], observed=True).dt.agg(["min","max"])
base = raw.dt.dt.to_period("M").min()
for col in ["min","max"]:
    assert ((pd.PeriodIndex(chk[col], freq="M") - base).map(lambda x: x.n)
            == chk.index.get_level_values(0)).all()
print("G1 PASS  every edge's activity lies within its labelled month")

tr, va, te = [sorted(SPLIT[k]) for k in ("train","val","test")]
assert max(tr) < min(va) and max(va) < min(te)
assert set(tr)|set(va)|set(te) == set(periods)
print("G2 PASS  splits chronological, disjoint, covering all", len(periods), "periods")

tv = [c for c in FEATURE_COLS
      if d[["customer_idx"]].merge(customer_features[["customer_idx", c]], on="customer_idx")
          .groupby("customer_idx")[c].nunique().max() > 1]
assert not tv, f"time-varying feature columns: {tv}"
print("G3 PASS  every model-facing feature is static")

def historical_affinity(edges, t, window=3):
    """Leakage-safe affinity features as of period t. Reads only period_idx <= t."""
    past = edges[edges.period_idx <= t]
    if past.empty:
        return pd.DataFrame(columns=["customer_idx","item_idx","lifetime_count","recency",
                                     "recent_count","item_share"])
    f = (past.groupby(["customer_idx","item_idx"], observed=True)
             .agg(lifetime_count=("period_idx","size"), last_seen=("period_idx","max")).reset_index())
    f["recency"] = t - f.last_seen
    rc = (past[past.period_idx > t - window].groupby(["customer_idx","item_idx"], observed=True)
              .size().reset_index(name="recent_count"))
    f = f.merge(rc, on=["customer_idx","item_idx"], how="left")
    f["recent_count"] = f.recent_count.fillna(0).astype(int)
    f["item_share"] = (f.lifetime_count /
                       f.groupby("customer_idx").lifetime_count.transform("sum")).round(4)
    return f.drop(columns="last_seen")

t_demo = 12; key = ["customer_idx","item_idx"]
a = historical_affinity(interactions, t_demo)
b = historical_affinity(interactions[interactions.period_idx <= t_demo], t_demo)
assert a.sort_values(key).reset_index(drop=True).equals(b.sort_values(key).reset_index(drop=True))
print(f"G4 PASS  historical_affinity(t={t_demo}) unchanged when future periods are deleted "
      f"({len(a):,} rows)")

G1 PASS  every edge's activity lies within its labelled month
G2 PASS  splits chronological, disjoint, covering all 24 periods


G3 PASS  every model-facing feature is static


G4 PASS  historical_affinity(t=12) unchanged when future periods are deleted (279,454 rows)


## 7. Export (same filenames and columns as dunnhumby)

In [8]:
transactions_clean = (d.rename(columns={"Invoice":"basket_id","Customer_ID":"customer_id",
                                        "StockCode":"item_label","Country":"country"})
    [["basket_id","customer_idx","customer_id","period_idx","dt","item_label","item_idx",
      "item_node_id","quantity","price","revenue","country"]]
    .sort_values(["period_idx","customer_idx","basket_id"]).reset_index(drop=True))

ARTIFACTS = {"transactions_clean": transactions_clean,
             "interactions_monthly": interactions,
             "customer_features": customer_features,
             "customer_id_map": customer_id_map,
             "item_id_map": item_id_map,
             "calendar": calendar}
for name, frame in ARTIFACTS.items():
    frame.to_csv(OUT/f"{SLUG}__{name}.csv", index=False)

META = {"dataset":"Online Retail II (UCI)", "slug":SLUG,
        "item_level":"StockCode", "item_meaning":"product SKU",
        "n_customers":int(N_CUST), "n_items":int(N_ITEM), "n_periods":int(len(periods)),
        "period_length":"1 calendar month", "calendar_type":"absolute (Dec 2009 - Nov 2011)",
        "cycle_len":CYCLE_LEN, "item_node_offset":int(N_CUST),
        "n_edges":int(len(interactions)), "default_edge_weight":"binary",
        "split":{k:list(v) for k,v in SPLIT.items()},
        "feature_columns":FEATURE_COLS,
        "not_features":["customer_id","country","country_grp","has_demographics",
                        "first_period_idx","last_period_idx","n_active_periods"],
        "feature_coverage":0.0,
        "feature_note":"Country only, 90% United Kingdom - treat as no usable node features"}
json.dump(META, open(OUT/f"{SLUG}__metadata.json","w"), indent=2)

print(pd.DataFrame([{"file":f"{SLUG}__{n}.csv","rows":len(f),"cols":f.shape[1],
                     "MB":round((OUT/f"{SLUG}__{n}.csv").stat().st_size/1024**2,1)}
                    for n,f in ARTIFACTS.items()]).to_string(index=False))
print(f"\n+ {SLUG}__metadata.json   ->  {OUT.resolve()}")

                                      file   rows  cols   MB
  online_retail_ii__transactions_clean.csv 749476    12 59.3
online_retail_ii__interactions_monthly.csv 702450     9 21.8
   online_retail_ii__customer_features.csv   5824    17  0.4
     online_retail_ii__customer_id_map.csv   5824     2  0.1
         online_retail_ii__item_id_map.csv   4618     5  0.2
            online_retail_ii__calendar.csv     24     7  0.0

+ online_retail_ii__metadata.json   ->  clean


## 8. Check

In [9]:
R = {n: pd.read_csv(OUT/f"{SLUG}__{n}.csv") for n in ARTIFACTS}
I, F, T, C = R["interactions_monthly"], R["customer_features"], R["transactions_clean"], R["calendar"]
act = {p:set(g.customer_idx) for p,g in I.groupby("period_idx")}; ps = sorted(act)
sup = sum(len(act[a]&act[b]) for a,b in zip(ps,ps[1:]))
prs = {p:set(zip(g.customer_idx,g.item_idx)) for p,g in I.groupby("period_idx")}
hit = tot = 0
for a,b in zip(ps,ps[1:]):
    both = act[a]&act[b]; x = {q for q in prs[a] if q[0] in both}
    hit += len(x & {q for q in prs[b] if q[0] in both}); tot += len(x)
seen = set().union(*[act[p] for p in SPLIT["train"]])
cold = np.mean([len(act[p]-seen)/len(act[p]) for p in list(SPLIT["val"])+list(SPLIT["test"])])

checks = {
 "no nulls in the model-facing artifacts": not I.isna().any().any() and not C.isna().any().any()
                                           and not F[["customer_idx"]+list(META["feature_columns"])].isna().any().any(),
 "customer_idx contiguous from 0":     sorted(F.customer_idx) == list(range(len(F))),
 "edge indices resolve to real nodes": I.customer_idx.max() < len(F) and I.item_idx.max() < len(R["item_id_map"]),
 "weight is binary everywhere":        (I.weight == 1).all(),
 "periods 0..23 present":              sorted(I.period_idx.unique()) == list(range(24)),
 "every period has a calendar row":    set(I.period_idx) == set(C.period_idx),
 "id maps round-trip":                 T.merge(R["customer_id_map"], on="customer_idx").eval("customer_id_x == customer_id_y").all(),
 "customers 5,824":                    len(F) == 5824,
 "items 4,618":                        len(R["item_id_map"]) == 4618,
 "edges 702,450":                      len(I) == 702450,
 "supervision 8,960":                  sup == 8960,
 "repeat rate 0.239":                  round(hit/tot, 3) == 0.239,
 "test window 616 customers":          len(act[22] & act[23]) == 616,
 "cold start 20.4%":                   round(cold*100, 1) == 20.4,
}
for k, v in checks.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())

print(f"\n{len(raw):,} raw rows -> {len(T):,} line items ({len(T)/len(raw)*100:.1f}% kept) "
      f"-> {len(I):,} edges over {len(ps)} periods")

  PASS  no nulls in the model-facing artifacts
  PASS  customer_idx contiguous from 0
  PASS  edge indices resolve to real nodes
  PASS  weight is binary everywhere
  PASS  periods 0..23 present
  PASS  every period has a calendar row
  PASS  id maps round-trip
  PASS  customers 5,824
  PASS  items 4,618
  PASS  edges 702,450
  PASS  supervision 8,960
  PASS  repeat rate 0.239
  PASS  test window 616 customers
  PASS  cold start 20.4%

1,067,371 raw rows -> 749,476 line items (70.2% kept) -> 702,450 edges over 24 periods


Output: `clean/online_retail_ii__*.csv` and the metadata json, the same six files and columns as for dunnhumby.